# BraTS-Africa 3D-to-2D Slice Extraction

Extracts 2D axial T1c slices from the 3D BraTS-Africa NIfTI volumes. For each patient, the slices with the largest segmented tumour area are selected.

In [ ]:
# ------------------------------------------------------------
# Configuration
# Point DATA_ROOT at your raw BraTS-Africa NIfTI directory
# (one sub-folder of .nii.gz volumes per patient) and
# OUTPUT_ROOT at the directory for extracted 2D slices.
# ------------------------------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')

DATA_ROOT    = "PATH/TO/BRATS_AFRICA_NIFTI"   # per-patient .nii.gz volumes
OUTPUT_ROOT  = "PATH/TO/OUTPUT/glioma"        # extracted 2D T1c slices
NUM_PATIENTS = 30                             # number of patients to process
SLICES_PER_PATIENT = 5                        # top-N largest-tumour-area slices

# Data Preparation: 3D-to-2D BraTS Extraction Pipeline

**Objective:**
To validate the MobileNetV2 model (trained on 2D Kaggle images) on the external **BraTS-Africa dataset**, the 3D volumetric NIfTI data must be adapted into a 2D format compatible with the model.


### Mount Google Drive

In [ ]:
# Mount your Google Drive and update the path below
# from google.colab import drive
# drive.mount('/content/drive')

DATA_DIR = '/path/to/your/dataset'  # Update this

### 1. Import Necessary Libraries
We use `nibabel` for handling NIfTI medical images and `cv2` (OpenCV) for image resizing and saving.

In [ ]:
import os
import glob
import numpy as np
import nibabel as nib  # For reading .nii.gz files
import cv2             # For resizing and saving as .png
import matplotlib.pyplot as plt
from tqdm import tqdm  # For progress bars

### 2. Configuration & Paths
We define the source directory containing the raw BraTS-Africa NIfTI files and the target directory for the converted 2D images.
* **IMG_SIZE:** Set to `(224, 224)` to match the MobileNetV2 input resolution.
* **SLICES_PER_PATIENT:** We extract the **top 5** slices with the largest tumor area per patient to create a balanced validation set (Target: 30 patients × 5 slices = 150 images).

### 3. Image Normalization Strategy
1.  **Outlier Clipping:** We clip the top 1% of pixel intensities (`p99`) to remove scanner artifacts.
2.  **Scaling:** We normalize the remaining intensity range to **0-255** (uint8), ensuring the tumor tissue remains distinct against the background.

In [ ]:
def normalize_slice(slice_data):
    """Normalizes MRI intensity to 0-255 (uint8) with outlier clipping.
    This helps the tumor stand out against the background.
    """# Clip top 1% outliers (often scanner noise)
    p99 = np.percentile(slice_data, 99)
    slice_data = np.clip(slice_data, 0, p99)

    img_min = slice_data.min()
    img_max = slice_data.max()

    if img_max - img_min == 0:
        return slice_data.astype(np.uint8)

    # Min-Max Scaling to 0-255
    normalized = (slice_data - img_min) / (img_max - img_min)
    return (normalized * 255).astype(np.uint8)

### 4. 3D-to-2D Extraction Pipeline
This function handles the core data curation logic

In [ ]:
def process_gliomas():
    os.makedirs(OUTPUT_ROOT, exist_ok=True)

    # 1. Get Patient List
    patient_dirs = sorted([d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d))])

    selected_patients = patient_dirs[:NUM_PATIENTS]
    print(f"Selecting first {len(selected_patients)} patients from {len(patient_dirs)} total.")

    total_images = 0

    for pat_id in selected_patients:
        pat_path = os.path.join(DATA_ROOT, pat_id)

        # --- THE FIX IS HERE ---
        # We construct the filename using the patient ID folder name
        t1c_filename = f"{pat_id}-t1c.nii.gz"
        seg_filename = f"{pat_id}-seg.nii.gz"
        t1c_path = os.path.join(pat_path, t1c_filename)
        seg_path = os.path.join(pat_path, seg_filename)

        if not os.path.exists(t1c_path) or not os.path.exists(seg_path):
            print(f"Skipping {pat_id}: Files missing.\n  Expected: {t1c_path}")
            continue

        try:
            # 3. Load 3D Volumes
            t1c_vol = nib.load(t1c_path).get_fdata()
            seg_vol = nib.load(seg_path).get_fdata()
        except Exception as e:
            print(f"Error loading {pat_id}: {e}")
            continue

        # 4. Find the 'Best' Slices (largest tumor area)
        slice_scores = []
        depth = seg_vol.shape[2]

        for z in range(depth):
            tumor_area = np.sum(seg_vol[:, :, z] > 0)
            if tumor_area > 0:
                slice_scores.append((z, tumor_area))

        slice_scores.sort(key=lambda x: x[1], reverse=True)
        top_slices = slice_scores[:SLICES_PER_PATIENT]

        if not top_slices:
            print(f"No tumor mask found for {pat_id}")
            continue

        # 5. Extract, Resize, and Save
        for (z_index, area) in top_slices:
            img_slice = t1c_vol[:, :, z_index]

            # Rotate if needed (MRI scans often load sideways)
            img_slice = np.rot90(img_slice, k=1)

            img_norm = normalize_slice(img_slice)
            img_resized = cv2.resize(img_norm, IMG_SIZE, interpolation=cv2.INTER_CUBIC)

            filename = f"{pat_id}_slice{z_index}.png"cv2.imwrite(os.path.join(OUTPUT_ROOT, filename), img_resized)
            total_images += 1

        print(f"Processed {pat_id}: Saved {len(top_slices)} slices.")

    print(f"Done! Extracted {total_images} images to {OUTPUT_ROOT}")

if __name__ == "__main__":
    process_gliomas()

### 5. Verification
We perform a final count of the extracted images to ensure the validation set is balanced and complete.
* **Target:** 150 images (30 patients $\times$ 5 slices).

In [ ]:
folder_path = OUTPUT_ROOT

if os.path.exists(folder_path):
    # Count only PNG files to be precise
    file_count = len([f for f in os.listdir(folder_path) if f.endswith('.png')])
    print(f"Found {file_count} images in '{folder_path}'")

    # Sanity check: It should be 150 (30 patients * 5 slices)
    if file_count == 150:
        print("You have exactly 30 patients × 5 slices.")
    else:
        print(f"Warning: Expected 150 images, but found {file_count}.")
else:
    print(f"Error: The folder '{folder_path}' does not exist. Did the extraction script finish?")